# DS4 — Feature Extraction

**Dataset:** Dataset 4 — Large non-experimental data  
**Stage:** 1 — Dataset-specific feature extraction (no cross-dataset harmonization)  
**Input files:** `outputs/cleaned/ds4_*.csv`  
**Output files:** `outputs/features/ds4/`

---

## Outputs produced

| File | Rows | Contents |
|---|---|---|
| `ds4_behavior_features.csv` | ~1,039 | All behavioral participants |
| `ds4_survey_features.csv` | ~283 | Demographics + qualitative flags (no Likert scales in DS4) |

---

## Event definitions

- **What counts as an event row:** Every row in `ds4_behavioral_clean.csv`.  
  Three structural event classes exist — keep them separate:
  - `intervention_event` (36.5%): active friction shown; has interventionType + Duration
  - `app_close_event` (47.1%): closedApp outcome; no intervention fields
  - `no_intervention` (16.4%): opened/dismissed but no intervention logging
- **Intervention exposure:** `intervention_event_class == 'intervention_event'`
- **Outcome-bearing rows:** All rows with `event_outcome_simple` ∈ {opened, dismissed, closed}.  
  Additionally: intervention-event-specific rates use `intervention_event` class only.
- **Longitudinal axis:** `days_since_first_event` (all rows, no NaN).
  Do NOT use `secondsSinceStudySetup` — it is inverted and unusable.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

CLEANED = PROJECT_ROOT / "outputs" / "cleaned"
OUT = PROJECT_ROOT / "outputs" / "features" / "ds4"
OUT.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output folder:", OUT)

## 1. Load cleaned files

In [ ]:
from src.features.ds4_features import load_ds4_files

beh, survey, linkage = load_ds4_files(CLEANED)

print(f"Behavioral rows:         {len(beh):,}")
print(f"Behavioral participants:  {beh['participant_id_canonical'].nunique():,}")
print(f"Survey rows:             {len(survey):,}")
print(f"Linkage rows:            {len(linkage):,}")

# Event class breakdown
print("\nEvent class distribution:")
print(beh['intervention_event_class'].value_counts())

## 2. Build behavior features

Feature groups:
- **G1** Volume by event class (DIRECT, High)
- **G2** Outcome rates by class (DIRECT, High)
- **G3** Intervention type profile — 6 types (DIRECT, High — DS4-unique)
- **G4** Temporal structure (DIRECT-DERIVED, High)
- **G5** Longitudinal activity (DIRECT-DERIVED, High)
- **G6** Adaptation via calendar time (PROXY, Medium)
- **G7** App structure (DIRECT, High)
- **G8** Intervention duration (DIRECT/PROXY, Medium)
- **G9** Override tendency (PROXY, Medium)

⚠️ DS4 is the largest dataset (2.5M rows). This will take a few minutes.

In [ ]:
from src.features.ds4_features import build_ds4_behavior_features

print("Building behavioral features (may take several minutes for 2.5M rows)...")
beh_feats = build_ds4_behavior_features(beh)
print(f"Shape: {beh_feats.shape}")
beh_feats.head(3)

In [ ]:
# Validation
assert beh_feats['participant_id_canonical'].nunique() == len(beh_feats), \
    "Duplicate participant IDs"
print(f"Participants: {len(beh_feats):,}")
print("\nKey feature distributions:")
print(beh_feats[[
    'total_events', 'obs_window_days', 'active_weeks',
    'rate_opened_all', 'rate_dismissed_all', 'rate_closed_all',
    'rate_opened_intervention_events',
    'n_intervention_types_seen'
]].describe().round(3))

In [ ]:
# Intervention type coverage
print("Intervention type exposure rates (mean across participants with >= 1 intervention event):")
type_cols = [c for c in beh_feats.columns if c.endswith('_rate') and 'Exercise' in c or c.endswith('_rate') and any(
    t.lower() in c.lower() for t in ['breathing', 'dot', 'black', 'mirror', 'rotate', 'minimal'])]
# Simpler: use endswith _rate columns from DS4 types
from src.features.ds4_features import DS4_INTERVENTION_TYPES
type_rate_cols = [f"{t}_rate" for t in DS4_INTERVENTION_TYPES]
print(beh_feats[type_rate_cols].describe().round(4))

In [ ]:
# Longitudinal coverage
print("Observation window statistics:")
print(beh_feats['obs_window_days'].describe().round(1))
print(f"\nParticipants with adaptation features: {beh_feats['open_rate_change_calendar'].notna().sum():,}")

In [ ]:
out_path = OUT / "ds4_behavior_features.csv"
beh_feats.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(beh_feats):,} rows, {beh_feats.shape[1]} columns)")

## 3. Build survey features

⚠️ DS4 survey contains ONLY qualitative open-text. No psychometric scales.  
Structured features limited to: age, gender, qualitative availability flags.

Feature groups:
- **G8** Demographics + qualitative flags (DIRECT — demographics; flags only for text)

In [ ]:
from src.features.ds4_features import build_ds4_survey_features

survey_feats = build_ds4_survey_features(beh, survey)
print(f"Shape: {survey_feats.shape}")
print("\nAge distribution:")
print(survey_feats['survey_age'].describe().round(1))
print("\nGender distribution:")
print(survey_feats['gender_clean'].value_counts(dropna=False))
survey_feats.head(3)

In [ ]:
out_path = OUT / "ds4_survey_features.csv"
survey_feats.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(survey_feats):,} rows, {survey_feats.shape[1]} columns)")

## 4. Output summary

In [ ]:
print("=" * 60)
print("DS4 Feature Extraction — Output Summary")
print("=" * 60)

for fname, df in [
    ('ds4_behavior_features.csv', beh_feats),
    ('ds4_survey_features.csv', survey_feats),
]:
    n_features = df.shape[1] - 1
    print(f"  {fname}")
    print(f"    {len(df):,} participants × {n_features} features")
    missing_any = df.drop(columns='participant_id_canonical').isna().any(axis=1).sum()
    print(f"    Participants with any NaN feature: {missing_any:,}")
    print()